In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\AQI_daily_2024_Aya_Nagar_Delhi_IMD_2024.xlsx")

In [3]:
df

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,NaN,154.0,135.0,148.0,184.0,185.0,NaN,187.0,NaN,135.0,323.0,268.0
1,2,248.0,NaN,93.0,127.0,NaN,117.0,NaN,55.0,NaN,100.0,318.0,270.0
2,3,332.0,NaN,NaN,129.0,235.0,121.0,NaN,89.0,95.0,100.0,383.0,NaN
3,4,308.0,NaN,109.0,143.0,NaN,176.0,46.0,54.0,88.0,98.0,377.0,194.0
4,5,238.0,NaN,101.0,143.0,265.0,165.0,57.0,59.0,89.0,103.0,NaN,196.0
5,6,NaN,104.0,NaN,124.0,255.0,114.0,54.0,41.0,80.0,104.0,340.0,185.0
6,7,232.0,128.0,183.0,169.0,307.0,251.0,49.0,49.0,85.0,105.0,357.0,191.0
7,8,241.0,112.0,160.0,150.0,160.0,206.0,49.0,36.0,98.0,NaN,NaN,256.0
8,9,285.0,104.0,NaN,161.0,120.0,181.0,77.0,NaN,100.0,138.0,358.0,121.0
9,10,189.0,217.0,176.0,202.0,NaN,118.0,100.0,NaN,NaN,91.0,334.0,171.0


In [4]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape

(41, 13)

In [5]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))

In [6]:
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())

In [7]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,239.28125,136.368421,135.000000,148.0,184.000000,185.000000,63.653846,55.40625,84.26087,135.000000,323.00000,268.000000
1,2,248.00000,136.368421,93.000000,127.0,152.607143,117.000000,63.653846,55.00000,84.26087,100.000000,318.00000,270.000000
2,3,332.00000,136.368421,146.709677,129.0,152.607143,121.000000,63.653846,55.40625,95.00000,100.000000,383.00000,213.714286
3,4,308.00000,136.368421,109.000000,143.0,152.607143,176.000000,63.653846,54.00000,88.00000,98.000000,377.00000,194.000000
4,5,238.00000,136.368421,101.000000,143.0,152.607143,165.000000,57.000000,59.00000,89.00000,103.000000,304.83871,196.000000
5,6,239.28125,136.368421,146.709677,124.0,152.607143,114.000000,54.000000,41.00000,80.00000,104.000000,340.00000,185.000000
6,7,232.00000,136.368421,183.000000,169.0,152.607143,116.571429,63.653846,49.00000,85.00000,105.000000,357.00000,191.000000
7,8,241.00000,136.368421,160.000000,150.0,160.000000,206.000000,63.653846,36.00000,98.00000,166.764706,304.83871,256.000000
8,9,285.00000,136.368421,146.709677,161.0,120.000000,181.000000,77.000000,55.40625,100.00000,138.000000,358.00000,121.000000
9,10,189.00000,136.368421,176.000000,202.0,152.607143,118.000000,63.653846,55.40625,84.26087,91.000000,334.00000,171.000000
